In [2]:
import json
import pandas as pd
import pathlib
import ollama
import sys

In [2]:
df = pd.read_sql("SELECT timestamp_ms FROM telemetry_samples ORDER BY id", "sqlite:///track1.db")
# df.to_csv("telemetry_export.csv", index=False)
print(df.describe())

       timestamp_ms
count  3.194000e+03
mean   9.262427e+07
std    6.016762e+04
min    9.250858e+07
25%    9.257589e+07
50%    9.262578e+07
75%    9.267567e+07
max    9.272556e+07


In [29]:
df = pd.read_sql(
    """
    SELECT 
        timestamp_utc AS timestamp,
        acceleration_x,
        acceleration_y,
        acceleration_z,
        yaw,
        position_x,
        position_y,
        position_z,
        speed 
    FROM telemetry_samples
    WHERE distance_traveled != 0
    ORDER BY id
    """,
    "sqlite:///track2_good.db"
)

def preprocess(df):
    df['timestamp'] = pd.to_datetime(df["timestamp"], utc=True).astype("int64") // 10**6
    df['timestamp'] = (df['timestamp'] - min(df['timestamp'])) / 1000

    df['position_x'] = df['position_x'].astype(int)
    df['position_y'] = df['position_y'].astype(int)
    df['position_z'] = df['position_z'].astype(int)

    df['acceleration_x'] = df['acceleration_x'].astype(int)
    df['acceleration_y'] = df['acceleration_y'].astype(int)
    df['acceleration_z'] = df['acceleration_z'].astype(int)
    df['yaw'] = df['yaw'].round(decimals=2)
    df['speed'] = (df['speed'] * 3.6).astype(int)
    
    return df

df_fast = preprocess(df)

df_fast = df_fast.iloc[::19, :]
print(df_fast.shape)

(51, 9)


In [30]:
df2 = pd.read_sql(
    """
    SELECT 
        timestamp_utc AS timestamp,
        acceleration_x,
        acceleration_y,
        acceleration_z,
        yaw,
        position_x,
        position_y,
        position_z,
        speed 
    FROM telemetry_samples
    WHERE distance_traveled != 0
    ORDER BY id
    """,
    "sqlite:///track2_bad.db"
)

df_slow = preprocess(df2)

x = max(df_slow["timestamp"]) / max(df_fast["timestamp"])

df_slow = df_slow.iloc[::int(19*x), :]
print(df_slow.shape)

(51, 9)


In [35]:
df_combined = df_fast.copy()
for col in df_fast.columns:
    df_combined[col] = list(zip(df_fast[col], df_slow[col]))

In [34]:
records = df_combined.to_dict(orient="records")
# with open("segment_data2_comparison.json", "w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=2)

# print(f"{len(df_slow)} Einträge geschrieben nach segment_data.json")
text = str(records)
for char in '[]{}()':
    text = text.replace(char, '')
    
text = text.replace('timestamp', '\ntimestamp')

with open("output.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
with open('output.txt') as input_file:
    text = f'{input_file.readlines()}'

text = text.strip('"')
text.replace('\'', '')

system_prompt = "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten und gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung. Antworte kurz, fokussiert, praxisnah, maximal 2 Sätze. Timestamp ist immer in Sekunden. Jedes Attribut hat zwei Werte: Das zweite steht immer für die zu bewertende Runde, der erste Wert beschreibt eine optimale Runde die dir als Referenz dient. **Verwende dafür ausschließlich die Daten aus der Liste des Users**. Negative Prompt: Denk dir keine weiteren Daten aus, Bewerte nicht die ersten Werte der Attribute"
user_prompt = "Bewerte meine Fahrleistung, zeige mir klar die Unterschiede:"

resp = ollama.chat(
    model="nemotron-3-nano:30b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + f"\n```json\n{text}\n```"},
    ],
)

print(resp["message"]["content"])
pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
with open("prompts/prompts.txt", "a", encoding="utf-8") as file:
    file.writelines([
        "System Prompt: " + str(system_prompt) + "\n",
        "User Prompt: " + str(user_prompt) + "\n",
        "Response: " + str(resp["message"]["content"]) + "\n\n",
    ])

Deine Z‑Beschleunigung liegt zwischen 1 und 6 m/s², während das optimale Referenzband (7‑23 m/s²) deutlich höher ist – das verursacht eine geringere Kurvenausfahrt‑Geschwindigkeit. Gleichzeitig übersteigt die aktuelle Höchstgeschwindigkeit den Referenzwert bereits stark, sodass mehr Gas früher eingesetzt werden sollte, um daraus Mehr­zeit zu gewinnen.
